# Task 1: Repository and Kaggle Setup


## Helper Functions

In [8]:
import os, json, subprocess
from kaggle_secrets import UserSecretsClient

TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = "https://github.com/Ahmad-Gill/qwen2.5-sft-dpo-alignment.git"
REPO_DIR = "/kaggle/working/qwen2.5-sft-dpo-alignment"


def run(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((out.stdout + out.stderr).replace(TOKEN, "***"))
    if out.returncode != 0:
        raise RuntimeError("Command failed")


def setup_repo():
    if not os.path.isdir(REPO_DIR):
        run(f"git clone {REPO_URL.replace('https://', f'https://{TOKEN}@')} {REPO_DIR}")
    os.chdir(REPO_DIR)
    run("git pull")
    run('git config user.name "kaggle" && git config user.email "kaggle@users.noreply.github.com"')


def install():
    run("pip uninstall -y -q torchao || true")
    run('pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" '
        "huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest")
    run("pip install -q -e . --no-deps")


def save_and_push(data, path, msg):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    run(f'git add results notebooks && (git diff --cached --quiet || '
        f'(git commit -q -m "{msg}" && git push -q))')
    print(" Saved and pushed:", path)

## main calling functions

In [9]:
setup_repo()
install()

import torch, transformers, datasets
gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]

Already up to date.







## results

In [10]:
info = {
    "gpus": gpus,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
}
print(json.dumps(info, indent=2))

if gpus:
    run("nvidia-smi")
    assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
else:
    print("⚠️ CPU test mode. Switch to GPU T4 x2 for the final run.")

save_and_push(info, "results/task1_setup/setup_info.json",
              f"Task 1 setup ({'GPU' if gpus else 'CPU'})")

{
  "gpus": [
    "Tesla T4",
    "Tesla T4"
  ],
  "torch": "2.11.0+cu128",
  "transformers": "4.57.6",
  "datasets": "4.8.5"
}
Sun Oct 11 11:20:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|          